# DubbingStory — Colab/Kaggle Local Vision (Qwen3-VL)

Notebook ini menjalankan pipeline DubbingStory dengan **Qwen3-VL-2B-Instruct** melalui `vLLM` lokal, sehingga analisis visual tidak membutuhkan API vision berbayar.

Alur notebook dibuat sama untuk Google Colab dan Kaggle. Perbedaan platform hanya ada pada cara mengambil `GOOGLE_API_KEY` dari Secrets.


## 1. Clone Repo

Download source code project dari branch `main`.


In [ ]:
# Clone repository langsung ke current directory agar tidak nested
!rm -rf ./* ./.[!.]* ./..?* 2>/dev/null || true
!git clone -b main https://github.com/NaufalRizqullah/dubbingstory.git .


## 2. Konfigurasi Pipeline

Pilih mode pipeline:
- `full` → dubbing seluruh video
- `summary` → membuat highlight recap dari scene terpenting


In [ ]:
# --- SETTINGS ---
VIDEO_INPUT = "https://www.youtube.com/watch?v=ms4wRkLIO5U"  # Bisa URL atau path file lokal
RESOLUTION = "1080"
PROJECT_NAME = "my_dubbing_project"
STYLE = "viral_fb"
LANGUAGE = "id"
RATIO = "16:9"
ENGINE_TTS = "edge"

# --- PIPELINE MODE ---
MODE = "summary"               # "full" atau "summary"
SUMMARY_DURATION = 60 * 3      # Target durasi ringkasan (detik), None = otomatis
SUMMARY_MAX_SCENES = None      # Maks scene, None = otomatis

# --- SPEED / SCENE SETTINGS ---
MAX_KEYFRAMES = 3
MIN_SCENE_DURATION = 3.0
SCENE_THRESHOLD = 4.0

# --- VISION MODEL ---
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

# Bounded output untuk analisis per-scene.
# Retry/rescue di repo menggunakan budget lebih kecil lagi jika model mengalami runaway output.
VISION_MAX_TOKENS = "640"

# --- IMAGE MODE ---
# "file" = kirim path lokal ke vLLM (lebih ringan untuk Colab/Kaggle)
# "data" = kirim base64 di request
IMAGE_MODE = "file"
LOCAL_MEDIA_ROOT = __import__("os").path.abspath(".")

# --- PATCH PREFLIGHT ---
# Fail fast jika branch main yang baru di-clone belum membawa patch repo utama.
from pathlib import Path

_cli_src = Path("dubbingstory/cli.py").read_text(encoding="utf-8")
_vision_src = Path("dubbingstory/vision/openai_vision.py").read_text(encoding="utf-8")

assert "cfg.mode = mode" in _cli_src, (
    "Repo belum membawa fix summary-mode propagation (cfg.mode = mode). "
    "Apply/commit patch repo utama lalu push ke branch main."
)
assert "scene_max_tokens = min(int(self.max_tokens), 640)" in _vision_src, (
    "Repo belum membawa bounded per-scene generation fix. "
    "Apply/commit patch repo utama lalu push ke branch main."
)
print("✅ Patch preflight passed: summary flow + bounded scene generation aktif")

# --- COLAB/KAGGLE GPU AUTO-DETECT ---
try:
    import torch
    NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    NUM_GPUS = 0

if NUM_GPUS >= 1:
    TENSOR_PARALLEL_SIZE = "1"
    MAX_MODEL_LEN = "12288"
    GPU_MEM_UTIL = "0.85"
    print(f"🖥️ Detected {NUM_GPUS} GPU(s) -> tensor-parallel-size=1, max-model-len=12288")
else:
    TENSOR_PARALLEL_SIZE = "1"
    MAX_MODEL_LEN = "8192"
    GPU_MEM_UTIL = "0.80"
    print("⚠️ No GPU detected — aktifkan GPU accelerator sebelum melanjutkan.")


## 3. Download Video & Subtitles

Video diunduh lebih dulu sebelum instalasi dependensi berat. Cookie YouTube bersifat opsional dan dideteksi otomatis.


In [ ]:
# Optional cookies: satu cell yang sama untuk Kaggle maupun Colab.
import os
import shutil

USE_COOKIES = True
COOKIES_PATH = os.path.abspath("youtube_cookies.txt")

# Kaggle: dataset cookie yang biasa digunakan.
# Colab: upload youtube_cookies.txt ke working directory (/content) bila diperlukan.
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        shutil.copy(source_cookie, COOKIES_PATH)
    print(f"✅ YouTube cookies aktif: {COOKIES_PATH}")
else:
    USE_COOKIES = False
    print("ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.")


In [ ]:
import subprocess
import time
import urllib.request
import json
import sys
import os

print("   - Install Deno (JS runtime) + ffmpeg untuk membantu yt-dlp")
try:
    subprocess.check_call(
        ["apt-get", "-qq", "update"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["apt-get", "-qq", "install", "-y", "ffmpeg", "curl", "unzip"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["mkdir", "-p", "/root/.deno/bin"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        [
            "curl", "-L", "--retry", "5", "--retry-all-errors", "--connect-timeout", "20",
            "-o", "/tmp/deno.zip",
            "https://github.com/denoland/deno/releases/latest/download/deno-x86_64-unknown-linux-gnu.zip",
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["unzip", "-o", "/tmp/deno.zip", "-d", "/root/.deno/bin"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["chmod", "+x", "/root/.deno/bin/deno"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    os.environ["PATH"] += ":/root/.deno/bin"
    subprocess.check_call(
        ["deno", "--version"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    print("   ✅ Deno terinstall dan PATH updated")
except Exception as e:
    print(f"   ⚠️ Gagal install Deno + ffmpeg (opsional): {e}")

print("   - Upgrade yt-dlp ke versi terbaru")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "yt-dlp"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)


In [ ]:
import sys
import os
sys.path.append(".")

from dubbingstory.ingest.youtube import download_video, download_subtitles

print("📥 Memulai proses download video...")
PROJECT_DIR = os.path.join("outputs", PROJECT_NAME)
os.makedirs(PROJECT_DIR, exist_ok=True)

if "http" in VIDEO_INPUT:
    try:
        download_kwargs = {
            "download_height": RESOLUTION,
        }
        if USE_COOKIES:
            download_kwargs["cookies"] = COOKIES_PATH

        video_path = download_video(
            VIDEO_INPUT,
            PROJECT_DIR,
            **download_kwargs,
        )

        # Subtitle opsional untuk konteks tambahan model.
        download_subtitles(VIDEO_INPUT, PROJECT_DIR)
        print("\n✅ Download SUKSES!")
    except Exception as e:
        print(f"\n❌ DOWNLOAD GAGAL: {e}")
        print("\nJANGAN lanjutkan ke cell berikutnya. Periksa URL/cookies lalu coba lagi.")
        raise
else:
    print("ℹ️ VIDEO_INPUT bukan URL. Dianggap sebagai file lokal.")


## 4. Install Dependencies & Environment Optimization

Setelah download video sukses, instal dependensi project dan vLLM.


In [ ]:
# Install dependencies utama project
!pip install -q -r requirements.txt openai

import subprocess
import sys
import os

print("🔧 Fixing environment...")
print("   - Uninstall torchaudio (vLLM tidak membutuhkannya untuk pipeline ini)")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    check=False,
)

# --- DETEKSI KEMAMPUAN GPU ---
try:
    import torch
    if torch.cuda.is_available():
        cc_major, cc_minor = torch.cuda.get_device_capability()
    else:
        cc_major, cc_minor = 0, 0
except Exception:
    cc_major, cc_minor = 0, 0

if cc_major > 0 and cc_major < 7:
    print(f"   - Detected older GPU (Compute Capability {cc_major}.{cc_minor}).")
    print("   - Install PyTorch 2.7.1 (cu126) untuk kompatibilitas GPU lama.")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--force-reinstall",
            "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
            "--index-url", "https://download.pytorch.org/whl/cu126",
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
else:
    print(f"   - Detected modern GPU (Compute Capability {cc_major}.{cc_minor}).")
    print("   - Install torch + torchvision sinkron CUDA 13.0.")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
            "torch==2.13.0", "torchvision==0.28.0",
            "--index-url", "https://download.pytorch.org/whl/cu130",
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )

print("   - Upgrade vLLM")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "vllm"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)

print("✅ Done. Lanjut ke setup secret lalu jalankan pipeline.")


## 5. Setup API Key

DubbingStory menggunakan Gemini API untuk penyusunan narasi/script. Model vision tetap berjalan lokal melalui vLLM.

Simpan secret dengan nama `GOOGLE_API_KEY` pada platform notebook yang digunakan.


In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os

try:
    API_KEY_GEMINI = UserSecretsClient().get_secret("GOOGLE_API_KEY") or ""
except Exception as e:
    raise RuntimeError("Kaggle Secret GOOGLE_API_KEY tidak ditemukan / belum diberi akses.") from e

env_text = f"""# Auto-generated from notebook secrets
GOOGLE_API_KEY={API_KEY_GEMINI}
OPENAI_VISION_IMAGE_MODE={IMAGE_MODE}
OPENAI_VISION_MAX_TOKENS={VISION_MAX_TOKENS}
OPENAI_VISION_MODEL_MAX_CONTEXT={MAX_MODEL_LEN}
"""

Path(".env").write_text(env_text, encoding="utf-8")
os.environ["GOOGLE_API_KEY"] = API_KEY_GEMINI
print("✅ .env dibuat dari Kaggle Secrets")


## 6. Jalankan Pipeline

Server vLLM dijalankan sebagai subprocess, lalu output pipeline di-stream ke notebook dan disimpan ke `pipeline.log`.


In [ ]:
import subprocess
import time
import urllib.request
import json
import sys
import os
from dotenv import load_dotenv

load_dotenv()

def wait_for_server(url, timeout=600):
    print(f"\n⏳ Waiting for vLLM server at {url} (timeout: {timeout}s)...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            req = urllib.request.Request(f"{url}/models")
            with urllib.request.urlopen(req) as response:
                if response.status == 200:
                    data = json.loads(response.read().decode())
                    print(f"\n✅ vLLM Server ready. Models: {[m['id'] for m in data['data']]}")
                    return True
        except Exception:
            pass
        sys.stdout.write(".")
        sys.stdout.flush()
        time.sleep(5)

    print("\n❌ Timeout waiting for vLLM server.")
    return False

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY belum tersedia. Jalankan cell Setup API Key terlebih dahulu.")

print(f"🚀 Starting vLLM server with model: {MODEL_NAME}...")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--port", str(PORT),
    "--max-model-len", MAX_MODEL_LEN,
    "--tensor-parallel-size", TENSOR_PARALLEL_SIZE,
    "--gpu-memory-utilization", GPU_MEM_UTIL,
    "--dtype", "half",
    "--enforce-eager",
]

if IMAGE_MODE == "file":
    vllm_cmd.extend(["--allowed-local-media-path", LOCAL_MEDIA_ROOT])

print(f"   Command: {' '.join(vllm_cmd)}")

vllm_log = open("vllm_server.log", "w")
vllm_process = subprocess.Popen(
    vllm_cmd,
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
)

try:
    if not wait_for_server(BASE_URL):
        raise RuntimeError("vLLM gagal start. Periksa vllm_server.log.")

    print(f"\n🚀 Starting DubbingStory Pipeline (mode: {MODE})...")

    local_video_path = os.path.join("outputs", PROJECT_NAME, "source.mp4")
    if "http" in VIDEO_INPUT and os.path.exists(local_video_path):
        cmd_input = local_video_path
        print(f"Menggunakan video yang sudah didownload: {cmd_input}")
    else:
        cmd_input = VIDEO_INPUT

    cmd = [
        sys.executable, "-u", "main.py", "run",
        "--input", cmd_input,
        "--project", PROJECT_NAME,
        "--style", STYLE,
        "--lang", LANGUAGE,
        "--ratio", RATIO,
        "--mode", MODE,
        "--vision-provider", "openai",
        "--vision-model", MODEL_NAME,
        "--vision-base-url", BASE_URL,
        "--vision-max-tokens", VISION_MAX_TOKENS,
        "--engine", ENGINE_TTS,
        "--max-keyframes", str(MAX_KEYFRAMES),
        "--min-scene-duration", str(MIN_SCENE_DURATION),
        "--scene-threshold", str(SCENE_THRESHOLD),
    ]

    if MODE == "summary":
        if SUMMARY_DURATION is not None:
            cmd.extend(["--summary-duration", str(SUMMARY_DURATION)])
        if SUMMARY_MAX_SCENES is not None:
            cmd.extend(["--summary-max-scenes", str(SUMMARY_MAX_SCENES)])

    if "http" in cmd_input:
        cmd[cmd.index("--input")] = "--url"
        cmd.append("--i-have-rights")

    print(f"Executing: {' '.join(cmd)}")

    pipeline_log_path = "pipeline.log"
    with open(pipeline_log_path, "w") as plog:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            plog.write(line)
        returncode = proc.wait()

    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)

    print(f"\n🎉 Pipeline completed successfully! (mode: {MODE})")
    print(f"📂 Output: outputs/{PROJECT_NAME}/")

except Exception as e:
    print(f"\n❌ An error occurred: {e}")

    for log_name in ("pipeline.log", "vllm_server.log"):
        if os.path.exists(log_name) and os.path.getsize(log_name) > 0:
            print(f"\n──── tail of {log_name} ────")
            with open(log_name, "r", errors="replace") as lf:
                print("".join(lf.readlines()[-80:]))
    raise

finally:
    print("\n🛑 Shutting down vLLM server...")
    vllm_process.terminate()
    try:
        vllm_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        vllm_process.kill()
        vllm_process.wait()
    vllm_log.close()
